In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import os
import sys

project_root = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "project_paths.py").is_file()),
    None,
)
if project_root is None:
    raise RuntimeError("프로젝트 폴더에서 노트북을 실행하세요.")
if str(project_root) in sys.path:
    sys.path.remove(str(project_root))
sys.path.insert(0, str(project_root))
sys.modules.pop("project_paths", None)
from project_paths import DATA_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, PATIENT_MAPPING_PATH, deidentified_path

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta
import numpy as np

# 경로 설정
table_dir = str(PROCESSED_DIR / 'AEGIS-Flowsheet-Value/de')

# 환자별 노이즈 정보
patient_mapping_df = pd.read_csv(str(PATIENT_MAPPING_PATH), sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
patient_mapping_df['new_pt_id'] = patient_mapping_df['new_pt_id'].astype(str)

patient_id_map = patient_mapping_df.set_index('pt_id')['new_pt_id'].to_dict()
date_offset_map = patient_mapping_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_offset_map = patient_mapping_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(table_dir, '**/*.csv'), recursive = True)
print(f"[INFO] 총 처리 대상 파일 수 : {len(csv_files)}")


In [ ]:
# 컬럼 삭제
print("\n[INFO] 컬럼 제거")

columns_to_drop = [
    'cat_text'
]

for file in csv_files:
    try:
        flowsheet_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        total_cols_to_remove = len(columns_to_drop)
        present_columns = [col for col in columns_to_drop if col in flowsheet_df.columns]
        removed_cols_count = len(present_columns)

        if removed_cols_count > 0:
            flowsheet_df.drop(columns = present_columns, inplace = True)

        flowsheet_df.to_csv(file, index=False, encoding = 'utf-8')

        print(f"[INFO] {file} - 제거 대상: {total_cols_to_remove}개, 실제 제거: {removed_cols_count}개")

    except Exception as e:
        print(f"[ERROR] hos_dt 오류: {file} → {e}")


In [ ]:
# 1. pt_id 치환
print("\n[INFO] 1. pt_id 치환 시작")
for file in csv_files:
    try:
        flowsheet_df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')

        if 'pt_id' in flowsheet_df.columns:
            flowsheet_df['pt_id'] = flowsheet_df['pt_id'].map(patient_id_map).fillna(flowsheet_df['pt_id'])

        print(flowsheet_df.head(3))

        flowsheet_df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"[INFO] pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] pt_id 오류: {file} → {e}")


In [ ]:
# 2. hos_dt_first ±7일 ±30분
print("\n[INFO] 2. hos_dt_first 노이즈 적용")
for file in csv_files:
    try:
        flowsheet_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'hos_dt_first' in flowsheet_df.columns:
            flowsheet_df['hos_dt_first'] = pd.to_datetime(flowsheet_df['hos_dt_first'], errors='coerce')
            flowsheet_df['hos_dt_first'] = flowsheet_df['hos_dt_first'] + flowsheet_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(flowsheet_df.head(3))

            flowsheet_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] hos_dt_first 저장 완료: {file}")

    except Exception as e:
        print(f"[ERROR] hos_dt_first 오류: {file} → {e}")


In [ ]:
# 2. cat_dt ±7일 ±30분
print("\n[INFO] cat_dt 노이즈 적용")
for file in csv_files:
    try:
        flowsheet_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')

        if 'cat_dt' in flowsheet_df.columns:
            flowsheet_df['cat_dt'] = pd.to_datetime(flowsheet_df['cat_dt'], errors='coerce')
            flowsheet_df['cat_dt'] = flowsheet_df['cat_dt'] + flowsheet_df['pt_id'].map(lambda x: timedelta(days=date_offset_map.get(x, 0), minutes=time_offset_map.get(x, 0)))

            print(flowsheet_df.head(3))

            flowsheet_df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"[INFO] cat_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[ERROR] cat_dt 오류: {file} → {e}")


In [ ]:
import numpy as np
# cat_vlaue 처리 (text 제거)
summary = []

for file in csv_files:
    try:
        flowsheet_df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        if 'cat_value' not in flowsheet_df.columns:
            print(f"[WARN] cat_value 컬럼 미존재 -> {file}")
            continue

        total_rows = len(flowsheet_df)

        print(f"\n[INFO] 처리 전 데이터 -> {os.path.basename(file)}")
        print(flowsheet_df.head(3))

        # Y,N, 수치형 보존
        keep_mask = (
            flowsheet_df['cat_value'].str.upper().isin(['Y','N']) |
            flowsheet_df['cat_value'].str.match(r'^-?\d+(\.\d+)?$', na=False)
        )

        #원본 보존
        original_nonnull = flowsheet_df['cat_value'].notna().sum()

        #text 결측치로 대체
        flowsheet_df.loc[~keep_mask, 'cat_value'] = np.nan

        replaced_to_na = original_nonnull - flowsheet_df['cat_value'].notna().sum()
        replacement_rate = replaced_to_na / total_rows if total_rows > 0 else 0

        print(f"[INFO] 처리 후 샘플 -> {os.path.basename(file)}")
        print(flowsheet_df.head(3))

        flowsheet_df.to_csv(file, index = False, encoding = 'utf-8')
        summary.append({
            'file': os.path.basename(file),
            'original_rows': total_rows,
            'replaced_to_NaN': replaced_to_na,
            'replacement_rate(%)': round(replacement_rate * 100, 2)
        })

        print(f"[INFO] 처리 완료: {file} --> NaN 대체 {replaced_to_na}건, 비율 {replacement_rate:.2%}")

    except Exception as e:
        print(f"[ERROR] 오류 발생: {file} -> {e}")

transformation_summary_df = pd.DataFrame(summary)
print(transformation_summary_df.head(10))
